# Project Overview

This notebook loads preprocessed data, trains a Logistic Regression model, and generates predictions on the test set. Evaluation is deferred to a later sprint.

# Import Libraries

Import the minimal libraries required for data loading and model training.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
    precision_score,
    recall_score,
)

from sklearn.model_selection import train_test_splitRANDOM_STATE = 42


# Load Processed Data

Load the preprocessed train/test splits produced by the preprocessing notebook.

In [49]:
DATA_DIR = Path("..") / "data" / "processed"
PIPELINE_OUTPUT_DIR = DATA_DIR / "pipeline_outputs"

X_TRAIN_FILE = PIPELINE_OUTPUT_DIR / "X_train.csv"
X_TEST_FILE = PIPELINE_OUTPUT_DIR / "X_test.csv"
Y_TRAIN_FILE = PIPELINE_OUTPUT_DIR / "y_train.csv"
Y_TEST_FILE = PIPELINE_OUTPUT_DIR / "y_test.csv"

if all(path.exists() for path in [X_TRAIN_FILE, X_TEST_FILE, Y_TRAIN_FILE, Y_TEST_FILE]):
    X_train = pd.read_csv(X_TRAIN_FILE)
    X_test = pd.read_csv(X_TEST_FILE)
    y_train = pd.read_csv(Y_TRAIN_FILE).squeeze()
    y_test = pd.read_csv(Y_TEST_FILE).squeeze()
    print("Loaded pipeline outputs from:", PIPELINE_OUTPUT_DIR)
else:
    raise FileNotFoundError(
        "Expected preprocessed pipeline outputs in data/processed/pipeline_outputs/."
    )

# Drop identifier columns if they remain in the saved feature sets
for id_col in ["customerID", "customer_id", "id"]:
    if id_col in X_train.columns:
        X_train = X_train.drop(columns=[id_col])
    if id_col in X_test.columns:
        X_test = X_test.drop(columns=[id_col])

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

Loaded pipeline outputs from: ..\data\processed\pipeline_outputs
X_train shape: (5625, 30)
X_test shape: (1407, 30)
y_train shape: (5625,)
y_test shape: (1407,)


# Logistic Regression

Train a Logistic Regression classifier on the prepared training data and generate test predictions. Logistic Regression estimates the probability of a binary outcome by fitting a linear decision boundary on the input features. It is commonly used as a baseline classifier because it is stable, interpretable, and fast to train.

In [50]:
model = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("Number of training samples:", X_train.shape[0])
print("Number of testing samples:", X_test.shape[0])

Number of training samples: 5625
Number of testing samples: 1407


d:\cv projects\RetentionAI\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


# Predictions

The trained Logistic Regression model generates predictions for the test set and stores them for later evaluation.

In [51]:
print("Number of predictions:", len(y_pred))
y_pred[:10]

Number of predictions: 1407


array([0, 1, 0, 0, 0, 0, 0, 0, 1, 0])

# Model Evaluation

Calculate standard classification metrics for the test predictions. These metrics measure different aspects of performance and provide more insight than accuracy alone.

In [52]:
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Accuracy: 0.8031
Precision: 0.6456
Recall: 0.5749
F1 Score: 0.6082

Classification Report:
              precision    recall  f1-score   support

           0       0.85      0.89      0.87      1033
           1       0.65      0.57      0.61       374

    accuracy                           0.80      1407
   macro avg       0.75      0.73      0.74      1407
weighted avg       0.80      0.80      0.80      1407



# Confusion Matrix

Visualize the confusion matrix and explain the meaning of each result for churn prediction.

In [ ]:
cm = confusion_matrix(y_test, y_pred)

tn, fp, fn, tp = cm.ravel()

disp = ConfusionMatrixDisplay(confusion_matrix=cm)
fig, ax = plt.subplots(figsize=(6, 6))
disp.plot(ax=ax)
plt.title("Confusion Matrix")
plt.show()

print(f"True Positives (TP): {tp}")
print(f"True Negatives (TN): {tn}")
print(f"False Positives (FP): {fp}")
print(f"False Negatives (FN): {fn}")

- True Positives (TP): churn customers correctly identified as churn.
- True Negatives (TN): retained customers correctly identified as not churn.
- False Positives (FP): retained customers incorrectly labeled as churn.
- False Negatives (FN): churn customers incorrectly labeled as not churn.

In churn prediction, false negatives are especially costly because they represent customers who churned without being flagged for retention efforts. False positives also matter because they can waste retention resources on customers who would have stayed.